In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.metrics import classification_report, cohen_kappa_score, accuracy_score, mean_absolute_error
from sklearn.utils.class_weight import compute_class_weight
import os
import zipfile

PROCESSED_DIR = '../data/processed/'
df_train = pd.read_parquet(PROCESSED_DIR + 'train_cleaned.parquet')
df_valid = pd.read_parquet(PROCESSED_DIR + 'valid_cleaned.parquet')

df_train['label'] = df_train['label'].astype(np.int64)
df_valid['label'] = df_valid['label'].astype(np.int64)

datasets = DatasetDict({
    'train': Dataset.from_pandas(df_train[['Sentence_Normalized', 'label']]),
    'valid': Dataset.from_pandas(df_valid[['Sentence_Normalized', 'label']]),
})
print(f" Dữ liệu sẵn sàng! Train: {len(df_train)} | Valid: {len(df_valid)}")

✅ Dữ liệu sẵn sàng! Train: 54626 | Valid: 7310 | Test: 7286


In [ ]:
MODEL_NAME = "aubmindlab/bert-base-arabertv02"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples["Sentence_Normalized"], 
        padding="max_length", 
        truncation=True, 
        max_length=128 
    )

print("Đang Tokenize bằng AraBERTv2...")
tokenized_datasets = datasets.map(tokenize_function, batched=True)

# Format PyTorch Tensors
tokenized_datasets["train"].set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
tokenized_datasets["valid"].set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
print("Hoàn tất Tokenize!")

⏳ Đang Tokenize bằng AraBERTv2...


Map:   0%|          | 0/54626 [00:00<?, ? examples/s]

Map:   0%|          | 0/7310 [00:00<?, ? examples/s]

Map:   0%|          | 0/7286 [00:00<?, ? examples/s]

✅ Hoàn tất Tokenize!


In [ ]:
print("TÍNH TOÁN TRỌNG SỐ MẪU (ALPHA FOR FOCAL LOSS)...")
y_train = df_train['label'].values
cw = compute_class_weight('balanced', classes=np.arange(19), y=y_train)

# Clip class weights to avoid extreme values
cw_clipped = np.clip(cw, 0.5, 5.0)
cw_normalized = cw_clipped / cw_clipped.mean()
class_weights_tensor = torch.tensor(cw_normalized, dtype=torch.float32)

print("Class weights (Alpha):")
print(np.round(cw_normalized, 2))

print("KHỞI TẠO MÔ HÌNH VÀ LORA CHO FOCAL LOSS...")
# Initialize the model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=19
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["query", "value", "dense"] 
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

⚖️ TÍNH TOÁN TRỌNG SỐ MẪU (ALPHA FOR FOCAL LOSS)...
🔸 Class weights (Alpha):
[2.02 2.02 1.02 1.99 0.44 0.97 0.28 0.26 0.73 0.2  0.29 0.2  0.36 0.2
 0.58 1.35 2.02 2.02 2.02]

⚙️ KHỞI TẠO MÔ HÌNH VÀ LORA CHO FOCAL LOSS...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at aubmindlab/bert-base-arabertv02 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


trainable params: 2,398,483 || all params: 137,606,438 || trainable%: 1.7430


In [ ]:
class FocalLossTrainer(Trainer):
    def __init__(self, *args, alpha=None, gamma=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.alpha = alpha  
        self.gamma = gamma  

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits  
        device = logits.device
        
        # 1. Cross-Entropy Loss per sample
        ce_loss = F.cross_entropy(logits, labels, reduction='none')
        
        # 2. pt = exp(-ce_loss) to get the probability of the true class
        pt = torch.exp(-ce_loss)
        
        # 3. Focal Loss per sample: (1 - pt)^gamma * ce_loss
        focal_loss = ((1.0 - pt) ** self.gamma) * ce_loss
        
        # 4. Apply alpha
        if self.alpha is not None:
            alpha_t = self.alpha.to(device)[labels]
            loss = (alpha_t * focal_loss).mean()
        else:
            loss = focal_loss.mean()
            
        return (loss, outputs) if return_outputs else loss

def compute_metrics_focal(eval_pred):
    logits, labels = eval_pred
    pred_labels = np.argmax(logits, axis=-1)
    
    qwk = cohen_kappa_score(labels, pred_labels, weights='quadratic')
    acc = accuracy_score(labels, pred_labels)
    
    return {"qwk": qwk, "accuracy": acc}

In [ ]:
training_args = TrainingArguments(
    output_dir="../saved_models/arabert_lora_focal",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,               
    per_device_train_batch_size=8,    
    gradient_accumulation_steps=4,    
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="qwk",
    greater_is_better=True,
    bf16=True,
    fp16=False,  
    seed=42
)

trainer = FocalLossTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["valid"],
    compute_metrics=compute_metrics_focal,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    alpha=class_weights_tensor,
    gamma=2.0
)

print("BẮT ĐẦU HUẤN LUYỆN FOCAL LOSS (GAMMA = 2.0)...")
trainer.train()

trainer.save_model("../saved_models/arabert_lora_focal_best")
tokenizer.save_pretrained("../saved_models/arabert_lora_focal_best")

🚀 BẮT ĐẦU HUẤN LUYỆN FOCAL LOSS (GAMMA = 2.0)...


  0%|          | 0/8535 [00:00<?, ?it/s]

{'loss': 0.6454, 'grad_norm': 15.847980499267578, 'learning_rate': 0.00028242530755711773, 'epoch': 0.29}
{'loss': 0.4994, 'grad_norm': 9.061336517333984, 'learning_rate': 0.00026485061511423544, 'epoch': 0.59}
{'loss': 0.4412, 'grad_norm': 10.502086639404297, 'learning_rate': 0.00024727592267135325, 'epoch': 0.88}


  0%|          | 0/457 [00:00<?, ?it/s]

{'eval_loss': 0.4426870048046112, 'eval_qwk': 0.7585737598002061, 'eval_accuracy': 0.45006839945280436, 'eval_runtime': 30.2105, 'eval_samples_per_second': 241.969, 'eval_steps_per_second': 15.127, 'epoch': 1.0}
{'loss': 0.3924, 'grad_norm': 27.518909454345703, 'learning_rate': 0.000229701230228471, 'epoch': 1.17}
{'loss': 0.3667, 'grad_norm': 17.242277145385742, 'learning_rate': 0.00021212653778558875, 'epoch': 1.46}
{'loss': 0.3609, 'grad_norm': 6.969714641571045, 'learning_rate': 0.00019455184534270648, 'epoch': 1.76}


...

  0%|          | 0/457 [00:00<?, ?it/s]

{'eval_loss': 0.38739046454429626, 'eval_qwk': 0.7955927422944147, 'eval_accuracy': 0.5357045143638851, 'eval_runtime': 30.1992, 'eval_samples_per_second': 242.059, 'eval_steps_per_second': 15.133, 'epoch': 4.0}
{'loss': 0.2348, 'grad_norm': 4.1884870529174805, 'learning_rate': 5.39543057996485e-05, 'epoch': 4.1}
{'loss': 0.2117, 'grad_norm': 5.485196590423584, 'learning_rate': 3.6379613356766254e-05, 'epoch': 4.39}
{'loss': 0.2148, 'grad_norm': 5.891729831695557, 'learning_rate': 1.8804920913884008e-05, 'epoch': 4.69}
{'loss': 0.2028, 'grad_norm': 8.457548141479492, 'learning_rate': 1.2302284710017573e-06, 'epoch': 4.98}


  0%|          | 0/457 [00:00<?, ?it/s]

{'eval_loss': 0.4045581817626953, 'eval_qwk': 0.7958108062274674, 'eval_accuracy': 0.5351573187414501, 'eval_runtime': 30.2496, 'eval_samples_per_second': 241.656, 'eval_steps_per_second': 15.108, 'epoch': 5.0}
{'train_runtime': 3493.4103, 'train_samples_per_second': 78.184, 'train_steps_per_second': 2.443, 'train_loss': 0.32664264925893594, 'epoch': 5.0}


('../saved_models/arabert_lora_focal_best\\tokenizer_config.json',
 '../saved_models/arabert_lora_focal_best\\special_tokens_map.json',
 '../saved_models/arabert_lora_focal_best\\vocab.txt',
 '../saved_models/arabert_lora_focal_best\\added_tokens.json',
 '../saved_models/arabert_lora_focal_best\\tokenizer.json')

In [ ]:
print("ĐANG DỰ ĐOÁN TRÊN TẬP VALIDATION (FOCAL LOSS)...")
predictions_output = trainer.predict(tokenized_datasets["valid"])
logits = predictions_output.predictions  # Kích thước [7310, 19]
true_labels = predictions_output.label_ids.astype(int)

final_pred_labels = np.argmax(logits, axis=-1)

print("=== BÁO CÁO F1-SCORE FOCAL LOSS ===")
target_names = [f"Level_{i+1}" for i in range(19)]
print(classification_report(true_labels, final_pred_labels, target_names=target_names, zero_division=0))

print("=== CÁC CHỈ SỐ METRIC BAREC ===")
print(f" QWK (Main Metric)       : {cohen_kappa_score(true_labels, final_pred_labels, weights='quadratic'):.4f}")
print(f" Acc19 (Exact Match)     : {accuracy_score(true_labels, final_pred_labels):.4f}")
print(f" Adjacent Acc (±1 Level)  : {np.mean(np.abs(true_labels - final_pred_labels) <= 1):.4f}")
print(f" Avg Distance (MAE)      : {mean_absolute_error(true_labels, final_pred_labels):.4f}")

🔍 ĐANG DỰ ĐOÁN TRÊN TẬP VALIDATION (FOCAL LOSS)...


  0%|          | 0/457 [00:00<?, ?it/s]


📊 === BÁO CÁO F1-SCORE FOCAL LOSS ===
              precision    recall  f1-score   support

     Level_1       0.71      0.82      0.76        44
     Level_2       0.52      0.54      0.53        68
     Level_3       0.41      0.75      0.53       182
     Level_4       0.39      0.60      0.47        78
     Level_5       0.55      0.57      0.56       417
     Level_6       0.42      0.61      0.50       189
     Level_7       0.60      0.62      0.61       701
     Level_8       0.66      0.64      0.65       613
     Level_9       0.45      0.67      0.54       236
    Level_10       0.70      0.78      0.74      1012
    Level_11       0.34      0.42      0.38       409
    Level_12       0.54      0.35      0.43      1491
    Level_13       0.43      0.53      0.47       349
    Level_14       0.61      0.49      0.54      1072
    Level_15       0.28      0.25      0.27       258
    Level_16       0.19      0.18      0.18       114
    Level_17       0.24      0.45      0.3